In [56]:
import numpy as np
import igl
import meshplot as mp
import scipy as sp
from scipy.spatial.transform import Rotation
import ipywidgets as iw
import time


In [57]:
v, f = igl.read_triangle_mesh('data/hand.off')
labels = np.load('data/hand.label.npy').astype(int)
v -= v.min(axis=0)
v /= v.max()

In [58]:
handle_vertex_positions = v.copy()
pos_f_saver = np.zeros((labels.max() + 1, 6))
def pos_f(s,x,y,z, α, β, γ):
    slices = (labels==s)
    r = Rotation.from_euler('xyz', [α, β, γ], degrees=True)
    v_slice = v[slices] + np.array([[x,y,z]])
    center = v_slice.mean(axis=0)
    handle_vertex_positions[slices] = r.apply(v_slice - center) + center
    pos_f_saver[s - 1] = [x,y,z,α,β,γ]
    t0 = time.time()
    v_deformed = pos_f.deformer(handle_vertex_positions)
    p.update_object(vertices = v_deformed)
    t1 = time.time()
    if t1 - t0 > 0:
        print('FPS', 1/(t1 - t0))
pos_f.deformer = lambda x:x

In [59]:
def widgets_wrapper():
    segment_widget = iw.Dropdown(options=np.arange(labels.max()) + 1)
    translate_widget = {i:iw.FloatSlider(min=-1, max=1, value=0) 
                        for i in 'xyz'}
    rotate_widget = {a:iw.FloatSlider(min=-90, max=90, value=0, step=1) 
                     for a in 'αβγ'}

    def update_seg(*args):
        (translate_widget['x'].value,translate_widget['y'].value,
        translate_widget['z'].value,
        rotate_widget['α'].value,rotate_widget['β'].value,
        rotate_widget['γ'].value) = pos_f_saver[segment_widget.value]
    segment_widget.observe(update_seg, 'value')
    widgets_dict = dict(s=segment_widget)
    widgets_dict.update(translate_widget)
    widgets_dict.update(rotate_widget)
    return widgets_dict

In [ ]:
def position_deformer(target_pos):
    # By recommendation, this function uses cache to pre-factor the equation, so its not repeated during slider interaction.
    if not hasattr(position_deformer, "_cache"):
        L = (-igl.cotmatrix(v, f)).tocsc()

        M = igl.massmatrix(v, f, igl.MASSMATRIX_TYPE_VORONOI).tocsc()
        Minv = sp.sparse.diags(1.0 / M.diagonal())

        # Bi-Laplacian quadratic form
        Q = (L @ Minv @ L).tocsc()

        H = labels > 0  # H: handle vertices, don't deform
        R = ~H          # R: Not H, free vertices to deform

        Q_RR = Q[R][:, R].tocsc()
        Q_RH = Q[R][:, H].tocsc()

        # Pre-factor for fast repeated solves during slider interaction
        solve_RR = sp.sparse.linalg.factorized(Q_RR)
        position_deformer._cache = (H, R, Q_RH, solve_RR)

    H, R, Q_RH, solve_RR = position_deformer._cache

    targetV = target_pos.copy()

    # Solve independently for x,y,z:
    # Q_RR * v_R = -Q_RH * v_H
    for d in range(3):
        b = -(Q_RH @ targetV[H, d])
        targetV[R, d] = solve_RR(b)

    return targetV

pos_f.deformer = position_deformer

In [ ]:
## Widget UI

p = mp.plot(handle_vertex_positions, f, c=labels)
iw.interact(pos_f, **widgets_wrapper())

In [ ]:
S = v.copy()

# Base mesh B from S
B = position_deformer(S.copy())

# Deformed base mesh B' from posed handles
Bp = position_deformer(handle_vertex_positions.copy())

# Detail transfer B to S onto B' 
def row_normalize(x, eps=1e-12):
    n = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.maximum(n, eps)

def fallback_tangent_from_normal(n):
    # Robust tangent if projected edge degenerates
    axis = np.array([1.0, 0.0, 0.0]) if abs(n[0]) < 0.9 else np.array([0.0, 1.0, 0.0])
    t = np.cross(n, axis)
    t /= max(np.linalg.norm(t), 1e-12)
    return t

def choose_reference_edges(V, F, N):
    # For each vertex, pick outgoing edge with longest tangent-plane projection
    adj = igl.adjacency_list(F)
    ref_nbr = np.full(V.shape[0], -1, dtype=int)

    for i in range(V.shape[0]):
        nbrs = np.array(adj[i], dtype=int)
        if nbrs.size == 0:
            continue

        e = V[nbrs] - V[i]                         # outgoing edges
        proj = e - (e @ N[i])[:, None] * N[i]      # project to tangent plane
        lens = np.linalg.norm(proj, axis=1)
        ref_nbr[i] = nbrs[np.argmax(lens)]
    return ref_nbr

def build_frames(V, F, ref_nbr):
    # Frame columns: t1, t2, n
    N = igl.per_vertex_normals(V, F)
    N = row_normalize(N)

    t1 = np.zeros_like(V)
    t2 = np.zeros_like(V)

    for i in range(V.shape[0]):
        n = N[i]
        j = ref_nbr[i]

        if j < 0:
            t = fallback_tangent_from_normal(n)
        else:
            e = V[j] - V[i]
            t = e - np.dot(e, n) * n
            ln = np.linalg.norm(t)
            if ln < 1e-12:
                t = fallback_tangent_from_normal(n)
            else:
                t = t / ln

        b = np.cross(n, t)
        lb = np.linalg.norm(b)
        if lb < 1e-12:
            t = fallback_tangent_from_normal(n)
            b = np.cross(n, t)
            lb = np.linalg.norm(b)
        b = b / max(lb, 1e-12)

        # Re-orthogonalize t with n and b
        t = np.cross(b, n)
        t = t / max(np.linalg.norm(t), 1e-12)

        t1[i] = t
        t2[i] = b

    return t1, t2, N

# Frames on B and stable edge choices
NB = row_normalize(igl.per_vertex_normals(B, f))
ref_edge = choose_reference_edges(B, f, NB)

# Encode detail displacements from B to S in B-frames
t1_B, t2_B, n_B = build_frames(B, f, ref_edge)
d = S - B
c1 = np.sum(d * t1_B, axis=1)
c2 = np.sum(d * t2_B, axis=1)
c3 = np.sum(d * n_B, axis=1)

# Build frames on B' with same outgoing edges as B
t1_Bp, t2_Bp, n_Bp = build_frames(Bp, f, ref_edge)

# Reconstruct transferred displacement and final S'
d_prime = c1[:, None] * t1_Bp + c2[:, None] * t2_Bp + c3[:, None] * n_Bp
Sp = Bp + d_prime

# Plot S, B, B', S' in a 2x2 grid
sh = {"wireframe": False}

p = mp.subplot(S,  f, c=labels, s=[2, 2, 0], shading=sh)
mp.subplot(B,  f, c=labels, s=[2, 2, 1], data=p, shading=sh)
mp.subplot(Bp, f, c=labels, s=[2, 2, 2], data=p, shading=sh)
mp.subplot(Sp, f, c=labels, s=[2, 2, 3], data=p, shading=sh)

print("2x2 order: Top Left=S, Top Right=B, Bottom Left=B', Bottom Right=S'")